In [1]:
# ============================================================
# CELL 1: Config
# ============================================================
import os, re
import pandas as pd
from getpass import getuser

USER     = getuser()
DATA_DIR = f"C:/Users/{USER}/Documents/GitHub/tennis-homophily/data/atp"

# OLD_FILE is the previously accumulated Grand Slam file.
# To add a new season: set OLD_FILE = current merged file, NEW_FILE = new scraped file.
OLD_FILE = os.path.join(DATA_DIR, "grand_slam_matches_2018_2025.xlsx")
NEW_FILE = os.path.join(DATA_DIR, "grand_slam_matches_2024_2025_olympics.xlsx")
OUT_FILE = os.path.join(DATA_DIR, "grand_slam_matches_2018_2025.xlsx")

# Stage normalisation
# Old data (ATP website):  'Round of 64 ', 'Round of 32 ', 'Round of 16 ',
#                          'Quarter-Finals ', 'Semi-Finals ', 'Finals '
# New data (Wikipedia):    'First Round', 'Second Round', 'Third Round',
#                          'Quarter-Finals', 'Semi-Finals', 'Final'
GS_STAGE_MAP = {          # Australian Open, Roland Garros, Wimbledon (64-team draw)
    'First Round':    'Round of 64',
    'Second Round':   'Round of 32',
    'Third Round':    'Round of 16',
    'Quarter-Finals': 'Quarter-Finals',
    'Semi-Finals':    'Semi-Finals',
    'Final':          'Finals',
}
USO_STAGE_MAP = {         # US Open (64-team draw)
    'First Round':    'Round of 64',
    'Second Round':   'Round of 32',
    'Third Round':    'Round of 16',
    'Quarter-Finals': 'Quarter-Finals',
    'Semi-Finals':    'Semi-Finals',
    'Final':          'Finals',
}
OLYMPICS_STAGE_MAP = {    # Olympics (32-team draw, no Third Round, + Bronze Medal Match)
    'First Round':        'Round of 32',
    'Second Round':       'Round of 16',
    'Quarter-Finals':     'Quarter-Finals',
    'Semi-Finals':        'Semi-Finals',
    'Final':              'Finals',
    'Bronze Medal Match': 'Bronze Medal Match',
}

# Tournament metadata for new rows (codes are stable per tournament across years)
# Key: (tournament_label, year_as_str)
# Value: (tournament_code, location, date_string)
TOURNAMENT_META = {
    ("Australian Open", "2024"): (580, "Melbourne,Australia",     "14-28 Jan, 2024"),
    ("Roland Garros",   "2024"): (520, "Paris,France",            "26 May - 9 Jun, 2024"),
    ("Wimbledon",       "2024"): (540, "London,Great Britain",    "1-14 Jul, 2024"),
    ("US Open",         "2024"): (560, "New York,United States",  "26 Aug - 8 Sep, 2024"),
    ("Australian Open", "2025"): (580, "Melbourne,Australia",     "12-26 Jan, 2025"),
    ("Roland Garros",   "2025"): (520, "Paris,France",            "25 May - 8 Jun, 2025"),
    ("Wimbledon",       "2025"): (540, "London,Great Britain",    "30 Jun - 13 Jul, 2025"),
    ("US Open",         "2025"): (560, "New York,United States",  "25 Aug - 7 Sep, 2025"),
    ("Olympics",        "2021"): (None, "Tokyo,Japan",            "24 Jul - 1 Aug, 2021"),
    ("Olympics",        "2024"): (None, "Paris,France",           "27 Jul - 4 Aug, 2024"),
    # Gap-year completions scraped from Wikipedia
    ("US Open",         "2023"): (560, "New York,United States",  "28 Aug - 10 Sep, 2023"),
    ("US Open",         "2019"): (560, "New York,United States",  "26 Aug - 8 Sep, 2019"),
    # Tournament-years with incomplete ATP data → replaced by Wikipedia
    ("Australian Open", "2018"): (580, "Melbourne,Australia",     "15-28 Jan, 2018"),
    ("Roland Garros",   "2021"): (520, "Paris,France",            "30 May - 13 Jun, 2021"),
    ("US Open",         "2021"): (560, "New York,United States",  "30 Aug - 12 Sep, 2021"),
    ("Wimbledon",       "2021"): (540, "London,Great Britain",    "28 Jun - 11 Jul, 2021"),
    ("Australian Open", "2022"): (580, "Melbourne,Australia",     "17-30 Jan, 2022"),
    ("Roland Garros",   "2022"): (520, "Paris,France",            "22 May - 5 Jun, 2022"),
    ("Wimbledon",       "2022"): (540, "London,Great Britain",    "27 Jun - 10 Jul, 2022"),
    ("Wimbledon",       "2023"): (540, "London,Great Britain",    "3-16 Jul, 2023"),
}

print("Config loaded.")

Config loaded.


In [2]:
# ============================================================
# CELL 2: Load both files
# ============================================================
old = pd.read_excel(OLD_FILE)
new = pd.read_excel(NEW_FILE)

old = old.drop_duplicates(ignore_index=True)

print(f"Old : {old.shape}")
print(f"New : {new.shape}")
print(f"Old columns: {list(old.columns)}")

Old : (1997, 30)
New : (1198, 20)
Old columns: ['tournament', 'location', 'date', 'year', 'tournament_code', 'stage', 'match_duration', 'winners_p1', 'winners_p2', 'losers_p1', 'losers_p2', 'winners_set1', 'winners_set2', 'winners_set3', 'winners_set1_tiebreak', 'winners_set2_tiebreak', 'winners_set3_tiebreak', 'losers_set1', 'losers_set2', 'losers_set3', 'losers_set1_tiebreak', 'losers_set2_tiebreak', 'losers_set3_tiebreak', 'winners_set4', 'winners_set5', 'losers_set4', 'losers_set5', 'winners_set4_tiebreak', 'losers_set4_tiebreak', 'losers_set5_tiebreak']


In [3]:
# ============================================================
# CELL 2b: Drop stale/incomplete rows before re-merging
# ============================================================
# USO (2019/2021/2023/2024/2025): old ATP-website data has wrong stage labels or
#   partial coverage; name normalization can't reconcile abbreviations.
#   → Drop entirely; Wikipedia provides the definitive version.
#
# AO/RG 2024: Zhang Zhizhen's name is stored as "Z Zhang" (surname-first) in the
#   ATP website but as "Zhang Zhizhen" (given-first) on Wikipedia.  After first-
#   initial normalization both become "z. zhang" vs "z. zhizhen" respectively, so
#   the dedup misses one QF match in each tournament → +1 excess row.
#   → Drop old ATP rows; Wikipedia is complete and consistent.
#
# AO 2018/2022, RG 2021/2022, Wimbledon 2021/2022/2023:
#   ATP-website data is missing 1–2 matches per tournament (walkovers/byes not
#   recorded by the ATP website). Wikipedia brackets are complete.
#   → Drop old ATP rows; replace with Wikipedia.
#
# All other ATP data (pre-2024 non-listed) is kept.

REPLACE_WITH_WIKI = [
    ('US Open',         2019),
    ('US Open',         2021),
    ('US Open',         2023),
    ('US Open',         2024),
    ('US Open',         2025),
    ('Australian Open', 2018),   # 2 missing matches in ATP data
    ('Australian Open', 2022),   # 1 missing match in ATP data
    ('Australian Open', 2024),   # Zhang Zhizhen name dedup fix
    ('Roland Garros',   2021),   # 1 missing match in ATP data
    ('Roland Garros',   2022),   # 1 missing match in ATP data
    ('Roland Garros',   2024),   # Zhang Zhizhen name dedup fix
    ('Wimbledon',       2021),   # 1 missing match in ATP data
    ('Wimbledon',       2022),   # 1 missing match in ATP data
    ('Wimbledon',       2023),   # 1 missing match in ATP data
]
drop_mask = old.apply(
    lambda r: (r['tournament'], r['year']) in REPLACE_WITH_WIKI, axis=1
)
n_dropped = drop_mask.sum()
old = old[~drop_mask].reset_index(drop=True)
print(f"Dropped {n_dropped} rows from old file (replaced by Wikipedia).")
print(f"Old after cleanup: {old.shape}")

Dropped 882 rows from old file (replaced by Wikipedia).
Old after cleanup: (1115, 30)


In [4]:
# ============================================================
# CELL 3: Clean old data
# ============================================================
# Strip trailing spaces from stage names (all old stages end with ' ')
old['stage'] = old['stage'].str.strip()

print("Old stages after strip:", sorted(old['stage'].unique()))
print("Old tournaments:", sorted(old['tournament'].unique()))

Old stages after strip: ['1st Round Qualifying', '2nd Round Qualifying', 'Bronze Medal Match', 'Finals', 'Quarter-Finals', 'Round of 16', 'Round of 32', 'Round of 64', 'Semi-Finals']
Old tournaments: ['Australian Open', 'Olympics', 'Roland Garros', 'US Open', 'Wimbledon']


In [5]:
# ============================================================
# CELL 4: Clean new data
# ============================================================

# 4a. Fix HTML line-break separators in team names
# GS pages use this for all teams; Olympics scraper now handles flagIOCathlete
# correctly so brackets produce clean names, but the Infobox still uses <br/>.
def fix_br(s):
    if pd.isna(s):
        return s
    return re.sub(r'<br\s*/?>', ' / ', str(s), flags=re.IGNORECASE).strip()

for col in ['Team1', 'Team2', 'Winner']:
    new[col] = new[col].apply(fix_br)

# 4b. Map Wikipedia stage names to ATP-style Round names
uso_mask     = new['Tournament'] == 'US Open'
olympics_mask = new['Tournament'] == 'Olympics'

new['stage'] = new['Stage'].map(GS_STAGE_MAP)                          # default: 64-draw GS
new.loc[uso_mask,      'stage'] = new.loc[uso_mask,      'Stage'].map(USO_STAGE_MAP)
new.loc[olympics_mask, 'stage'] = new.loc[olympics_mask, 'Stage'].map(OLYMPICS_STAGE_MAP)
new['stage'] = new['stage'].fillna(new['Stage'])                        # fallback: keep original

print(f"New rows after loading: {len(new)}")
print(f"  of which Olympics: {olympics_mask.sum()}")
print("\nStage mapping preview:")
print(
    new[['Tournament', 'Stage', 'stage']]
    .drop_duplicates()
    .sort_values(['Tournament', 'stage'])
    .to_string(index=False)
)

New rows after loading: 1198
  of which Olympics: 64

Stage mapping preview:
     Tournament              Stage              stage
Australian Open              Final             Finals
Australian Open     Quarter-Finals     Quarter-Finals
Australian Open        Third Round        Round of 16
Australian Open       Second Round        Round of 32
Australian Open        First Round        Round of 64
Australian Open        Semi-Finals        Semi-Finals
       Olympics Bronze Medal Match Bronze Medal Match
       Olympics              Final             Finals
       Olympics     Quarter-Finals     Quarter-Finals
       Olympics       Second Round        Round of 16
       Olympics        First Round        Round of 32
       Olympics        Semi-Finals        Semi-Finals
  Roland Garros              Final             Finals
  Roland Garros     Quarter-Finals     Quarter-Finals
  Roland Garros        Third Round        Round of 16
  Roland Garros       Second Round        Round of 32
  Rol

In [6]:
# ============================================================
# CELL 5: Score-parsing helpers
# ============================================================

def parse_score_str(s):
    """
    '7-6(3)' or '7-6' → (7, 6).  Returns (None, None) if unparseable.
    Note: the tiebreak value in parentheses is already stored separately
    in TB_SetN and is not re-extracted here.
    """
    if pd.isna(s) or not str(s).strip():
        return None, None
    m = re.match(r'(\d+)-(\d+)', str(s))
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))


def reshape_row(row):
    """
    Convert one new-format row to the old ATP column schema.

    New format: score from Team1's perspective.
      Score_SetN = 'g1-g2'  (Team1 got g1, Team2 got g2)
      TB_SetN    = set loser's tiebreak points (standard tennis notation)

    Old schema:
      winners_setN / losers_setN  = game counts from winner / loser perspective
      winners_setN_tiebreak       = tiebreak pts when the match winner LOST that
                                    set's tiebreak (i.e. they were the set loser)
      losers_setN_tiebreak        = tiebreak pts when the match loser  LOST that
                                    set's tiebreak (i.e. they were the set loser)

    tournament_code, location, date are filled from TOURNAMENT_META.
    """
    t1_wins = str(row.get('Winner', '')).strip() == str(row.get('Team1', '')).strip()

    def split_pair(s):
        if pd.isna(s) or not str(s).strip():
            return None, None
        parts = [p.strip() for p in re.split(r'\s*/\s*', str(s))]
        return (parts[0] or None), (parts[1] if len(parts) > 1 else None)

    t1p1, t1p2 = split_pair(row['Team1'])
    t2p1, t2p2 = split_pair(row['Team2'])

    wp1, wp2 = (t1p1, t1p2) if t1_wins else (t2p1, t2p2)
    lp1, lp2 = (t2p1, t2p2) if t1_wins else (t1p1, t1p2)

    # Look up tournament metadata
    meta_key = (str(row['Tournament']), str(row['Year']))
    t_code, location, date_str = TOURNAMENT_META.get(meta_key, (None, None, None))

    out = {
        'tournament':      row['Tournament'],
        'location':        location,
        'date':            date_str,
        'year':            row['Year'],
        'tournament_code': t_code,
        'stage':           row['stage'],
        'match_duration':  None,
        'winners_p1':      wp1,
        'winners_p2':      wp2,
        'losers_p1':       lp1,
        'losers_p2':       lp2,
    }

    for n in range(1, 6):
        sc_raw = row.get(f'Score_Set{n}')
        tb_raw = row.get(f'TB_Set{n}')

        g1, g2 = parse_score_str(sc_raw)

        if g1 is None:
            out[f'winners_set{n}'] = None
            out[f'losers_set{n}']  = None
            out[f'winners_set{n}_tiebreak'] = None
            out[f'losers_set{n}_tiebreak']  = None
            continue

        wg = g1 if t1_wins else g2
        lg = g2 if t1_wins else g1
        out[f'winners_set{n}'] = wg
        out[f'losers_set{n}']  = lg

        tb = None if (pd.isna(tb_raw) if tb_raw is not None else True) else int(tb_raw)

        if tb is not None:
            if wg > lg:   # match winner won this set → set loser = match loser
                out[f'winners_set{n}_tiebreak'] = None
                out[f'losers_set{n}_tiebreak']  = tb
            else:         # match winner lost this set → set loser = match winner
                out[f'winners_set{n}_tiebreak'] = tb
                out[f'losers_set{n}_tiebreak']  = None
        else:
            out[f'winners_set{n}_tiebreak'] = None
            out[f'losers_set{n}_tiebreak']  = None

    return out

print("Helpers defined.")

Helpers defined.


In [7]:
# ============================================================
# CELL 6: Reshape new data rows
# ============================================================
records = [reshape_row(row) for _, row in new.iterrows()]
new_shaped = pd.DataFrame(records)
new_shaped = new_shaped.drop_duplicates(ignore_index=True)

print(f"Reshaped: {new_shaped.shape}")
print(new_shaped[['tournament','year','stage','winners_p1','winners_p2',
                   'losers_p1','losers_p2','winners_set1','losers_set1',
                   'winners_set1_tiebreak','losers_set1_tiebreak']].head(5).to_string())

Reshaped: (1198, 31)
        tournament  year        stage winners_p1    winners_p2  losers_p1   losers_p2  winners_set1  losers_set1  winners_set1_tiebreak  losers_set1_tiebreak
0  Australian Open  2018  Round of 64    Ł Kubot        M Melo  P Lorenzi    M Zverev           6.0          2.0                    NaN                   NaN
1  Australian Open  2018  Round of 64  M Purcell     L Saville    M Elgin    A Rublev           6.0          3.0                    NaN                   NaN
2  Australian Open  2018  Round of 64  F Fognini  M Granollers     A Bolt   B Mousley           6.0          7.0                    6.0                   NaN
3  Australian Open  2018  Round of 64      R Ram      D Sharan    M Copil   V Troicki           7.0          6.0                    NaN                   5.0
4  Australian Open  2018  Round of 64    F López       M López   F Mergea  N Zimonjić           6.0          2.0                    NaN                   NaN


In [8]:
# ============================================================
# CELL 7: Align columns and merge
# ============================================================

# The old schema has a specific column order (30 columns); we match it exactly.
# Note: old data has no 'winners_set5_tiebreak' column — reproduce this asymmetry.
OLD_COLS = list(old.columns)

for col in OLD_COLS:
    if col not in new_shaped.columns:
        new_shaped[col] = None

new_shaped = new_shaped[OLD_COLS]

# Normalization functions for deduplication
import re
import unicodedata

def normalize_player_name(name: str) -> str:
    if not name:
        return ""
    text = str(name)
    text = re.sub(r"\s*<br\s*/?>\s*", " / ", text, flags=re.IGNORECASE)
    text = re.sub(r"\[\[[^\]]+\]\]", lambda m: m.group(0).strip('[]'), text)
    text = re.sub(r"\{\{[^}]*\}\}", "", text)
    text = re.sub(r"[\(\[]\s*\d+\s*[\)\]]", "", text)
    text = re.sub(r"\bvs\.?\b", "", text, flags=re.IGNORECASE)
    text = text.replace('.', ' ')
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^\w\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    parts = text.split()
    if len(parts) == 0:
        return ""
    if len(parts) == 1:
        return parts[0]
    first = parts[0]
    last = parts[-1]
    if len(first) == 1:
        return f"{first}. {last}"
    return f"{first[0]}. {last}"

def normalize_team_name(name: str) -> str:
    if not name:
        return ""
    text = str(name)
    players = re.split(r"\s*/\s*|\s*<br\s*/?>\s*", text)
    players = [normalize_player_name(p) for p in players if p and p.strip()]
    players = sorted(p for p in players if p)
    return " / ".join(players)

# Apply normalization to merged dataframe for deduplication
merged = pd.concat([old, new_shaped], ignore_index=True)
merged['winners_p1_norm'] = merged['winners_p1'].map(normalize_player_name)
merged['winners_p2_norm'] = merged['winners_p2'].map(normalize_player_name)
merged['losers_p1_norm'] = merged['losers_p1'].map(normalize_player_name)
merged['losers_p2_norm'] = merged['losers_p2'].map(normalize_player_name)
merged['match_key'] = merged[['year','tournament','stage','winners_p1_norm','winners_p2_norm','losers_p1_norm','losers_p2_norm']].astype(str).agg('|'.join, axis=1)

before = len(merged)
merged = merged.drop_duplicates(subset='match_key', keep='first').reset_index(drop=True)
merged = merged.drop(columns=['winners_p1_norm','winners_p2_norm','losers_p1_norm','losers_p2_norm','match_key'])
print(f"Removed duplicates from merged dataframe: {before - len(merged)}")

# Sort: year asc, then tournament, then stage (stable keeps match order within stage)
STAGE_ORDER = {
    'Round of 64': 1, 'Round of 32': 2, 'Round of 16': 3,
    'Quarter-Finals': 4, 'Semi-Finals': 5, 'Finals': 6,
    '1st Round Qualifying': 0, '2nd Round Qualifying': 0,
}
merged['_stage_ord'] = merged['stage'].map(STAGE_ORDER).fillna(3)
merged = merged.sort_values(['year', 'tournament', '_stage_ord'], kind='stable')
merged = merged.drop(columns=['_stage_ord']).reset_index(drop=True)

print(f"Merged shape: {merged.shape}")

print()
print(merged.groupby(['year', 'tournament']).size().rename('matches').to_string())

Removed duplicates from merged dataframe: 316
Merged shape: (1997, 30)

year  tournament     
2018  Australian Open    63
      Roland Garros      63
      US Open            63
      Wimbledon          75
2019  Australian Open    63
      Roland Garros      63
      US Open            63
      Wimbledon          63
2020  Australian Open    63
      Roland Garros      63
      US Open            31
2021  Australian Open    63
      Olympics           32
      Roland Garros      63
      US Open            63
      Wimbledon          63
2022  Australian Open    63
      Roland Garros      63
      US Open            63
      Wimbledon          63
2023  Australian Open    63
      Roland Garros      63
      US Open            63
      Wimbledon          63
2024  Australian Open    63
      Olympics           32
      Roland Garros      63
      US Open            63
      Wimbledon          63
2025  Australian Open    63
      Roland Garros      63
      US Open            63
      Wimb

In [9]:
# ============================================================
# CELL 8: Validation
# ============================================================
print("=== Stage values in merged file ===")
print(sorted(merged['stage'].dropna().unique()))

print("\n=== Missing player names (new rows only) ===")
new_rows = merged[merged['year'] >= 2024]
print(f"  winners_p1 missing: {new_rows['winners_p1'].isna().sum()} / {len(new_rows)}")
print(f"  losers_p1  missing: {new_rows['losers_p1'].isna().sum()} / {len(new_rows)}")

print("\n=== Tiebreak coverage (new rows) ===")
tb_cols = [c for c in merged.columns if 'tiebreak' in c]
has_tb = new_rows[tb_cols].notna().any(axis=1)
print(f"  Matches with at least one tiebreak: {has_tb.sum()} / {len(new_rows)}")

print("\n=== Sample tiebreak rows (new data) ===")
sample = new_rows[has_tb][['tournament','year','stage','winners_p1','losers_p1',
                            'winners_set1','losers_set1','losers_set1_tiebreak',
                            'winners_set2','losers_set2','winners_set2_tiebreak',
                            'winners_set3','losers_set3','losers_set3_tiebreak']].head(8)
print(sample.to_string())

=== Stage values in merged file ===
['1st Round Qualifying', '2nd Round Qualifying', 'Bronze Medal Match', 'Finals', 'Quarter-Finals', 'Round of 16', 'Round of 32', 'Round of 64', 'Semi-Finals']

=== Missing player names (new rows only) ===
  winners_p1 missing: 0 / 536
  losers_p1  missing: 0 / 536

=== Tiebreak coverage (new rows) ===
  Matches with at least one tiebreak: 278 / 536

=== Sample tiebreak rows (new data) ===
           tournament  year        stage    winners_p1     losers_p1  winners_set1  losers_set1  losers_set1_tiebreak  winners_set2  losers_set2  winners_set2_tiebreak  winners_set3  losers_set3  losers_set3_tiebreak
1463  Australian Open  2024  Round of 64     S Bolelli     R Arneodo           3.0          6.0                   NaN           6.0          3.0                    NaN           7.0          6.0                   7.0
1465  Australian Open  2024  Round of 64     M Arévalo    C Frantzen           7.0          6.0                   5.0           6.0       

In [10]:
# ============================================================
# CELL 9: Save
# ============================================================
merged.to_excel(OUT_FILE, index=False)
print(f"Saved {len(merged)} rows → {OUT_FILE}")
print(f"  Old rows : {len(old)}")
print(f"  New rows : {len(new_shaped)}")

Saved 1997 rows → C:/Users/aldi/Documents/GitHub/tennis-homophily/data/atp\grand_slam_matches_2018_2025.xlsx
  Old rows : 1115
  New rows : 1198


In [11]:
# ============================================================
# CELL 10: Round-completeness audit
# ============================================================
EXPECTED_64 = {
    "Round of 64": 32, "Round of 32": 16, "Round of 16": 8,
    "Quarter-Finals": 4, "Semi-Finals": 2, "Finals": 1,
}
EXPECTED_OLY = {
    "Round of 32": 16, "Round of 16": 8, "Quarter-Finals": 4,
    "Semi-Finals": 2, "Finals": 1, "Bronze Medal Match": 1,
}
EXPECTED_2020_USO = {
    "Round of 32": 16, "Round of 16": 8, "Quarter-Finals": 4, "Semi-Finals": 2, "Finals": 1,
}

merged["stage"] = merged["stage"].str.strip()

rows = []
for (y, t), grp in merged.groupby(["year", "tournament"]):
    if t == "Olympics":
        expected = EXPECTED_OLY
    elif t == "US Open" and y == 2020:
        expected = EXPECTED_2020_USO
    else:
        expected = EXPECTED_64
    for stage, exp in expected.items():
        actual = len(grp[grp["stage"] == stage])
        rows.append({"year": y, "tournament": t, "stage": stage,
                     "actual": actual, "expected": exp})

grid = pd.DataFrame(rows)
grid["missing"] = grid["expected"] - grid["actual"]
incomplete = grid[grid["missing"] > 0].copy()

print("=== Incomplete rounds (actual < expected) ===")
print(incomplete[["year","tournament","stage","actual","expected","missing"]]
      .sort_values(["year","tournament","stage"]).to_string(index=False))

print()
print("=== Missing-match totals by tournament-year ===")
print(incomplete.groupby(["year","tournament"])["missing"].sum().to_string())

print()
print("Notes:")
print("  2020 USO            -> OK: COVID reduced draw (32 teams, no R64).")
print("  Single-match gaps   -> Likely walkovers or byes not entered in source data.")
print("  2018 AO R32/R16     -> One match known to not have been played (verified).")

=== Incomplete rounds (actual < expected) ===
Empty DataFrame
Columns: [year, tournament, stage, actual, expected, missing]
Index: []

=== Missing-match totals by tournament-year ===
Series([], )

Notes:
  2020 USO            -> OK: COVID reduced draw (32 teams, no R64).
  Single-match gaps   -> Likely walkovers or byes not entered in source data.
  2018 AO R32/R16     -> One match known to not have been played (verified).
